# ส่วนที่1 กระบวนการทำความสะอาดและเตรียมข้อมูลรีวิวภาษาไทย

## นำเข้าไลบรารี


In [ ]:
import pandas as pd

นำไฟล์จากการดึงข้อมูลเข้ามาทำความสะอาด

In [ ]:

try:
    from IPython.display import display
except:
    display = print

INPUT_FILE = "lazada30.csv"

print("1) โหลดไฟล์:", INPUT_FILE)
df = pd.read_csv(INPUT_FILE)
print("   จำนวนแถวทั้งหมด (raw):", len(df))
print("   คอลัมน์ทั้งหมด:", df.columns.tolist())
print("\n   ตัวอย่าง 5 แถวแรก (raw):")
display(df.head(5))



In [ ]:
# 1) แยกจำนวนข้อมูลตามปี
print("\n1) จำนวนข้อมูลตามปี (จากคอลัมน์ 'at')")
print("----------------------------------------------------")

df["at"] = pd.to_datetime(df["at"], errors="coerce")
df["year"] = df["at"].dt.year
df["month"] = df["at"].dt.month

# จำนวนข้อมูลตามปี
year_counts = df["year"].value_counts().sort_index()
print(year_counts)



คัดเลือกคอลัมน์และบันทึกข้อมูล


In [ ]:
# 2) เช็คข้อมูลซ้ำใน reviewId และ content 
print("\n2) ตรวจสอบข้อมูลซ้ำ")
print("----------------------------------------------------")

# --- เช็ค reviewId ซ้ำ ---
if "reviewId" in df.columns:
    dup_reviewId = df[df["reviewId"].duplicated(keep=False)]
    print(f"- จำนวน reviewId ซ้ำ: {dup_reviewId['reviewId'].nunique()} ค่า   (จำนวนแถวที่ซ้ำทั้งหมด: {len(dup_reviewId)})")
else:
    print("- ไม่พบคอลัมน์ reviewId")

# 3) ตรวจว่าคอลัมน์ content มีภาษาอังกฤษหรือไม่
print("\n3) ตรวจภาษาอังกฤษในคอลัมน์ 'content'")
print("----------------------------------------------------")

import re

def contains_english(text):
    if not isinstance(text, str):
        return False
    return bool(re.search(r"[A-Za-z]", text))

# ตรวจภาษาอังกฤษเฉพาะคอลัมน์ content
if "content" in df.columns:
    has_eng = df["content"].astype(str).apply(contains_english)
    total_eng_rows = has_eng.sum()
    print(f"- จำนวนแถวที่มีภาษาอังกฤษใน content: {total_eng_rows}")


In [ ]:
print("\n4) ลบแถวที่มีภาษาอังกฤษ")
print("----------------------------------------------------")

initial_rows = len(df)

# --- ลบแถวที่มีภาษาอังกฤษใน content ---
if "content" in df.columns:
    df = df[~df["content"].astype(str).apply(contains_english)]

final_rows = len(df)

print(f"- จำนวนแถวก่อนลบ: {initial_rows}")
print(f"- จำนวนแถวหลังลบ: {final_rows}")
print(f"- ลบออกทั้งหมด: {initial_rows - final_rows} แถว")


In [ ]:
# 1) แยกจำนวนข้อมูลตามปี (หลังจากลบข้อมูลซ้ำและภาษาอังกฤษแล้ว)
print("\n1) จำนวนข้อมูลตามปี (จากคอลัมน์ 'at') — หลังลบข้อมูลแล้ว")
print("----------------------------------------------------")

df["at"] = pd.to_datetime(df["at"], errors="coerce")
df["year"] = df["at"].dt.year
df["month"] = df["at"].dt.month

# จำนวนข้อมูลตามปี ก่อนลบปี 2023
year_counts = df["year"].value_counts().sort_index()

for yr, count in year_counts.items():
    print(f"ปี {int(yr)}: {count} แถว")

# ลบข้อมูลปี 2023
print("\nลบข้อมูลปี 2023 ออก...")
df = df[df["year"] != 2023]

# รีเซ็ต index หลังลบ
df = df.reset_index(drop=True)

# นับข้อมูลใหม่หลังลบปี 2023
print("\n1.1) จำนวนข้อมูลตามปี — หลังจากลบปี 2023 แล้ว")
print("----------------------------------------------------")

year_counts_after = df["year"].value_counts().sort_index()

for yr, count in year_counts_after.items():
    print(f"ปี {int(yr)}: {count} แถว")

# 2) แยกตามเดือนของแต่ละปี — หลังลบปี 2023
print("\n2) จำนวนข้อมูลแยกตามเดือนในแต่ละปี — หลังลบปี 2023")
print("----------------------------------------------------")

for yr in sorted(df["year"].dropna().unique()):
    print(f"\n=== ปี {int(yr)} ===")
    
    df_year = df[df["year"] == yr]
    month_counts = df_year["month"].value_counts().sort_index()

    # แสดงเดือนครบ 12 เดือน
    for m in range(1, 13):
        print(f"เดือน {m:02d}: {month_counts.get(m, 0)}")


บันทึกข้อมูลเพื่อนำไปใช้ต่อ

In [ ]:
output_file = "lazadaapp_cleaned.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nบันทึกไฟล์เรียบร้อยแล้ว -> {output_file}")
print("จำนวนแถวหลังคลีน:", len(df))


## ส่วนที่ 2 การเตรียมและสำรวจคำจากรีวิว 

ติดตั้งและ import ไลบรารี

In [ ]:
!pip install pythainlp emoji attacut -q

In [ ]:
pip install numpy>=2.0

In [ ]:
import re
import emoji
import pandas as pd
import ipywidgets as widgets
from collections import Counter
from pythainlp.corpus.common import thai_stopwords
from pythainlp.tokenize import word_tokenize
from IPython.display import display
from IPython.display import display, clear_output
from collections import Counter
import pandas as pd

อ่านไฟล์ lazadaapp_cleaned.csv และทำการแยกชุดข้อมูล
* 1) ชุดข้อมูลเต็ม (Full dataset) ใช้สำหรับงาน train ML / sentiment analysis
* 2) ชุดข้อมูลไม่ซ้ำ (Unique content) ใช้สำหรับ สำรวจคำและแยกประเด็น (Aspect)

In [ ]:
file_name = "lazadaapp_cleaned.csv"
df = pd.read_csv(file_name, encoding="utf-8")

print("โหลดไฟล์สำเร็จ")
print("จำนวนแถวทั้งหมด (full):", len(df))

# แยกชุดข้อมูล

# 1) ชุดเต็ม (ใช้ต่อ ML / sentiment)
df_full = df.copy()

# 2) ชุดไม่ซ้ำ (ใช้สำรวจคำ)
df_unique = (
    df[["content", "year", "month"]]
    .drop_duplicates(subset="content") #ลบแถวที่ content ซ้ำ
    .reset_index(drop=True)
)

print("จำนวนแถวหลังลบ content ซ้ำ (unique):", len(df_unique))


สร้างสำเนาจากชุดข้อมูลที่ไม่ซ้ำ (df_unique) เพื่อใช้ในการสำรวจคำในข้อความรีวิว (content)

In [ ]:
# ใช้เฉพาะการสำรวจคำ / จัดประเด็น
df_content = df_unique.copy()

# ตั้งค่า Pandas ให้แสดงข้อความเต็ม
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 10)

print("\nข้อมูลที่ใช้สำหรับการสำรวจคำ / จัดประเด็น")
print("จำนวนแถวทั้งหมด:", len(df_content))
print("จำนวนคอลัมน์:", len(df_content.columns))
print("ชื่อคอลัมน์:", list(df_content.columns))

print("\nตัวอย่างข้อมูลดิบ:")
display(df_content[["content"]].head(10))


ทำความสะอาดข้อความ (Text Cleaning)

* ทำความสะอาดข้อความรีวิว โดยลบอักขระที่ไม่เกี่ยวข้องกับการวิเคราะห์ภาษาไทย เช่น emoji, HTML tag, ตัวเลข และอักขระพิเศษ รวมถึงจัดการช่องว่างซ้ำ เน้น “ความสะอาดของคำ” เพื่อการสำรวจ keyword

In [ ]:
def clean_thai(text):
    if not isinstance(text, str):
        return ""

    # ลบ emoji
    text = emoji.replace_emoji(text, replace='')

    # ลบ HTML tag
    text = re.sub(r'<.*?>', ' ', text)

    # ลบตัวเลข (ทั้งเลขไทยและเลขอารบิก)
    text = re.sub(r'[0-9๐-๙]', ' ', text)

    # เก็บเฉพาะตัวอักษรไทย + ช่องว่าง
    text = re.sub(r'[^\u0E00-\u0E7F\s]', ' ', text)

    # ลบช่องว่างซ้ำ
    text = re.sub(r'\s+', ' ', text).strip()

    return text

การปรับรูปแบบคำ (Word Normalization)

* สำหรับลดความซ้ำซ้อนของตัวอักษร สระ และวรรณยุกต์ ซึ่งมักพบในรีวิวที่เขียนแบบไม่เป็นทางการ เช่น “ดีมากกก”, “ช้ามากกก”

In [ ]:
def normalize_word(word):
    # ลบช่องว่างหัว–ท้าย
    word = word.strip()

    # ลดสระที่ซ้ำ เช่น เเเ -> เ , าาา -> า , ััั -> ั
    word = re.sub(r'(เ|แ|โ|ใ|ไ)\1+', r'\1', word)
    word = re.sub(r'(า)\1+', r'\1', word)
    word = re.sub(r'(ั)\1+', r'\1', word)

    # ลดวรรณยุกต์ซ้ำ เช่น ่่่ -> ่ , ้้ -> ้ , ๊๊๊ -> ๊ , ๋๋๋ -> ๋
    word = re.sub(r'(่)\1+', r'\1', word)
    word = re.sub(r'(้)\1+', r'\1', word)
    word = re.sub(r'(๊)\1+', r'\1', word)
    word = re.sub(r'(๋)\1+', r'\1', word)

    return word

def remove_repeat_chars(word):
    return re.sub(r'(.)\1{2,}', r'\1', word)

การตัดคำภาษาไทย (Tokenization)

* ใช้ตัดคำภาษาไทยด้วย PyThaiNLP (attacut) จากนั้นทำ normalization เพิ่มเติม ลบคำซ้ำ คำที่ไม่ต้องการ และ stopwords เพื่อให้ได้ชุดคำที่เหมาะสมสำหรับการวิเคราะห์ประเด็น

In [ ]:
from pythainlp.corpus.common import thai_stopwords

stopwords = set(thai_stopwords())

def tokenize_thai(text):
    if not isinstance(text, str):
        return []

    # ตัดคำด้วย attacut
    words = word_tokenize(text, engine="attacut")

    # normalize คำ
    words = [normalize_word(w) for w in words]

    # ลบอักษรซ้ำ
    words = [remove_repeat_chars(w) for w in words]

    # ลบช่องว่าง
    words = [w for w in words if w.strip() != ""]

    # เก็บเฉพาะคำไทยล้วน
    words = [w for w in words if re.fullmatch(r'[ก-๙]+', w)]

    # blacklist คำไม่ต้องการ
    bad_words = {"ปก", "กก", "ๆๆ", "เเต่", "เเล้ว", "ใด้", "ชื้อ"}
    words = [w for w in words if w not in bad_words]

    # ลบคำที่มี ๆ ซ้ำ
    words = [w for w in words if not re.search(r'ๆ{2,}', w)]

    # ลบ stopwords และคำสั้น
    words = [w for w in words if w not in stopwords and len(w) > 1]

    return words

print(" ฟังก์ชัน tokenize พร้อมใช้งาน")


ตรวจสอบผลลัพธ์การทำความสะอาด

* แสดงตัวอย่างข้อความก่อนและหลังการทำความสะอาด รวมถึงผลการตัดคำ

In [ ]:
sample_df = df_content.head(10).copy()

sample_df["clean_text"] = sample_df["content"].apply(clean_thai)
sample_df["tokens"] = sample_df["clean_text"].apply(tokenize_thai)

print(" ตารางเปรียบเทียบผลลัพธ์:")
display(sample_df[["content", "clean_text", "tokens"]])


ประมวลผลทั้งชุดข้อมูล นำกระบวนการทำความสะอาดและตัดคำไปใช้กับข้อมูลรีวิวทั้งหมด

In [ ]:
df_content["clean_text"] = df_content["content"].apply(clean_thai)
df_content["tokens"] = df_content["clean_text"].apply(tokenize_thai)

print(" สร้าง tokens ให้ทั้งไฟล์เรียบร้อย")


การนับคำและสำรวจคำที่พบบ่อย

* รวมคำทั้งหมดจากรีวิวทุกข้อความเพื่อใช้วิเคราะห์ความถี่ของคำ
* แสดงคำที่พบบ่อยที่สุด 200 คำ เพื่อใช้เป็นข้อมูลประกอบการคัดเลือกคำสำคัญในขั้นตอนถัดไป

In [ ]:
all_tokens = [token for tokens in df_content["tokens"] for token in tokens]

common_words = Counter(all_tokens).most_common(200)

print(" Top 200 Tokens:")
common_words


สร้างตาราง df_words

* สร้างตารางความถี่ของคำในรูปแบบ DataFrame เพื่อความสะดวกในการนำไปจัดหมวดประเด็น

In [ ]:
word_counts = Counter(all_tokens)

df_words = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values(by="count", ascending=False)
      .reset_index(drop=True)
)

print(" df_words ถูกสร้างแล้ว:", df_words.shape)
display(df_words.head(20))


กำหนดหมวดประเด็นจากงานวิจัยเดิม อ้างอิงจากงานวิจัยของ Wanassawan และ Ekarat (2022) เรื่อง การวิเคราะห์ความคิดเห็นจากทวิตเตอร์ของลูกค้าบริษัทช้อปปี้ประเทศไทย

* กำหนดหมวดประเด็นและคำสำคัญตามงานวิจัย เพื่อใช้เป็นกรอบเริ่มต้นในการวิเคราะห์
* ตรวจสอบว่าคำจากงานวิจัยพบในข้อมูลจริงหรือไม่

In [ ]:
# ---------------------- หมวดหมู่จากวิจัย ----------------------
topics = {
    "System": [
        "แอป","แอปลาซาด้า","แอปพลิเคชั่น","หน้าจอ","รวน","สมัคร","ไถ",
        "ชั่วคราว","ระบบ","เงิน","หน้าเว็บ","เว็บ","ตัดเงิน","บัตร","แคนเซิล",
        "ยกเลิก","พัง","ค้าง","ล่ม","เสีย","ไม่ได้","เด้ง","รอ","พัฒนา",
        "หักเงิน","คืนเงิน","ตัดบัตร","ใช้ยาก","ใช้งานง่าย","ไม่ตรง","โอนเงิน",
        "เครดิต","แก้ไข","ขัดข้อง","ฟื้น","จ่ายเงิน","ผ่อน","ปัญหา","โดนระงับ",
        "ตรวจสอบ","ขั้นตอน","เงินไม่เข้า","เติมเงิน","จ่ายไม่ได้","แบบใหม่",
        "แบบเก่า","บัตรเครดิต","บอท","อัพเดต","เงินคืน","บัญชี","อนุมัติ",
        "กดเงิน","อัตโนมัติ","ฟีด","โหมดดาร์ก","สถานะ","แฮ็ค","ตะกร้า",
        "แจ้งเตือน","ลิ้งก์","สะดวก","ระบบล่ม","โหลด","ปัดจอ","เสิร์ช",
        "ธุรกรรม","วงเงิน","วอลเล็ต"
    ],

    "Logistics": [
        "ไม่ติดต่อ","ยกเลิก","รอ","ช้า","นาน","ไม่ได้รับ","ไม่ได้สินค้า","ปัญหา",
        "ผู้ส่ง","พนักงานขนส่ง","ขนส่ง","บริการ","ไม่โทรมา","ส่งของ","ไม่มาส่ง",
        "ไม่ส่ง","สุภาพ","บอกทาง","เช็ค","ตรวจ","ที่อยู่","ส่งช้า","ส่งเร็ว",
        "ส่งไว","โทรมา","ไม่โทร","ไม่รับ","เลือกขนส่ง","เลือกบริษัทขนส่ง",
        "ค่าส่ง","เก็บปลายทาง","เลขพัสดุ","จ่าหน้า"
    ],

    "Company": [
        "ลาซาด้า","บริษัท","แอดมิน","คอลเซ็นเตอร์","เจ้าหน้าที่","พนักงาน",
        "ค่าบริการ","รับผิดชอบ","ติดต่อ","โกงเงิน","บัญชี","ดำเนินการ",
        "คุยกับพนักงาน","บริการ","แจ้ง","พรีเมียม","เมมเบอร์","แบน",
        "เลิกใช้","สื่อสาร","ร้องเรียน","เว็บไซต์","เบอร์","โทร","เมล",
        "สนใจ","เพิกเฉย","รอสาย","ตัดสาย","ด่วน","ศูนย์บริการ","ชดเชย",
        "องค์กร","ประทับใจ","แนะนำ","ชี้แจง","นโยบาย","สมาชิก","การตลาด",
        "โฆษณา","ลูกค้า","ผู้ใช้บริการ","แพลตฟอร์ม"
    ],

    "Store": [
        "ร้าน","ร้านค้า","ช้อป","ซื้อ","สั่ง","จากลาซาด้า","ไลฟ์","ไลฟ์สด",
        "ราคา","คุ้ม","คุ้มค่า","ไม่คุ้ม","แพง","ถูก","ลด","คุณภาพ",
        "ส่วนต่าง","ควรซื้อ","ไปตำ","พิกัด","รีวิว","สอย","พร้อมส่ง",
        "คำสั่งซื้อ","แม่ค้า","พ่อค้า","โกง","ผู้ขาย","แกง","ร้านเสื้อผ้า",
        "เสียเงิน","ขายดี","หมดไว","ปลอม","สินค้า","ของ"
    ],

    "Promotion": [
        "โค้ด","โค๊ด","โค้ดลด","โค้ดลาซาด้า","กดไม่ทัน","ใช้ไม่ทัน","ค่าส่ง",
        "ลดราคา","ส่วนลด","โปรโมชั่น","โปร","โปรลด","ส่งฟรี","โค้ดส่งฟรี",
        "ใช้โค้ดไม่ได้","ขั้นต่ำ","ฟรี","แถม","เต็ม","แฟลชเซล","เมมเบอร์",
        "สิทธิ์","คอยน์","แคมเปญ","เหรียญ","ดีล","คูปอง","กิจกรรม"
    ]
}



found_missing_report = {}
for topic, keywords in topics.items():
    found_words = []
    missing_words = []
    for kw in keywords:
        if kw in all_tokens:    #แยกคำที่ “พบ” และ “ไม่พบ” ในแต่ละประเด็น
            found_words.append(kw)  
        else:
            missing_words.append(kw)
    
    found_missing_report[topic] = {"found": found_words, "missing": missing_words}

for topic, result in found_missing_report.items():
    print(f"\n====================")
    print(f"หมวด: {topic}")
    print("====================")
    print("คำที่พบในข้อมูล:")
    if result["found"]:
        print(result["found"])
    else:
        print(" - ไม่มีคำที่พบ -")

    print("\n คำที่ไม่พบในข้อมูล:")
    if result["missing"]:
        print(result["missing"])
    else:
        print(" - ไม่มีคำที่ขาด -")


คัดเลือกเฉพาะคำจากงานวิจัยที่พบจริงในข้อมูล
* คัดเลือกเฉพาะคำสำคัญจากงานวิจัยที่พบในข้อมูลจริง เพื่อนำมาใช้เป็นชุดคำเริ่มต้นในการจัดหมวดประเด็น

In [ ]:
# ----------------- Lazada Topic (คำที่คัดแล้วว่าเจอในข้อมูลจริงที่อ้างอิงจากงานวิจัย) ---------------- -
lazada_topics = {
    "System": [
        'แอป', 'แอปลาซาด้า', 'แอปพลิเคชั่น', 'รวน', 'สมัคร',
        'ไถ', 'ชั่วคราว', 'ระบบ', 'เงิน', 'เว็บ', 'บัตร', 'ยกเลิก',
        'พัง', 'ค้าง', 'ล่ม', 'เด้ง', 'รอ', 'พัฒนา', 'เครดิต',
        'ขัดข้อง', 'ปัญหา', 'โดนระงับ', 'ตรวจสอบ', 'ขั้นตอน',
        'บอท', 'อัพเดต', 'บัญชี', 'อนุมัติ', 'อัตโนมัติ', 'ฟีด',
        'สถานะ', 'ตะกร้า', 'สะดวก', 'โหลด', 'เสิร์ช', 'ธุรกรรม',
        'วงเงิน', 'วอลเล็ต'],

    "Logistics": [
        'ยกเลิก', 'รอ', 'ปัญหา', 'ขนส่ง', 'บริการ',
        'สุภาพ', 'เช็ค', 'ตรวจ', 'ส่งไว', 'เลขพัสดุ'
    ],

    "Company": [
        'ลาซาด้า', 'บริษัท', 'แอดมิน', 'เจ้าหน้าที่', 'พนักงาน',
        'รับผิดชอบ', 'ติดต่อ', 'บัญชี', 'ดำเนินการ', 'บริการ',
        'แจ้ง', 'พรีเมียม', 'แบน', 'สื่อสาร', 'ร้องเรียน', 'เว็บไซต์',
        'เบอร์', 'โทร', 'เมล', 'สนใจ', 'เพิกเฉย', 'ด่วน',
        'ชดเชย', 'องค์กร', 'ประทับใจ', 'แนะนำ', 'ชี้แจง', 'นโยบาย',
        'สมาชิก', 'โฆษณา', 'ลูกค้า', 'แพลตฟอร์ม'],

    "Store": [
        'ร้าน', 'ร้านค้า', 'ช้อป', 'ซื้อ', 'สั่ง', 'ไลฟ์', 'ราคา',
        'คุ้ม', 'แพง', 'ลด', 'คุณภาพ', 'พิกัด', 'รีวิว', 'สอย',
        'แม่ค้า', 'พ่อค้า', 'โกง', 'แกง', 'ปลอม', 'สินค้า'
    ],

    "Promotion": [
        'โค้ด', 'โค๊ด', 'โปรโมชั่น', 'โปร', 'โปรลด', 'ฟรี', 'แถม',
        'เต็ม', 'สิทธิ์', 'คอยน์', 'แคมเปญ', 'เหรียญ', 'ดีล', 'คูปอง', 'กิจกรรม'
    ]
}


วิเคราะห์คำอื่น ๆ จากข้อมูลจริงที่อยู่นอกกรอบวิจัย
* วิเคราะห์คำที่พบบ่อยในข้อมูลรีวิว เพื่อค้นหาคำเพิ่มเติมที่อาจสะท้อนประเด็นสำคัญจากมุมมองผู้ใช้จริง
* จัดกลุ่มคำตามหมวดประเด็น เพื่อดูการกระจายของคำในแต่ละประเด็น
* ตรวจสอบคำที่ยังไม่ถูกจัดหมวด เพื่อพิจารณาว่าสามารถเชื่อมโยงกับประเด็นใดได้บ้าง

In [ ]:
word_counts = Counter(all_tokens)

df_words = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values(by="count", ascending=False)
      .reset_index(drop=True)
)

print(" df_words created:", df_words.shape)

# ---------------- ผูกคำกับหมวด ----------------
df_words['lazada_topics'] = df_words['word'].apply(
    lambda w: next(
        (t for t, kws in lazada_topics.items() if w in kws),
        "อื่นๆ"
    )
)

# ---------------- Interactive widget ----------------
thai_topics = list(lazada_topics.keys()) + ["อื่นๆ"]

multi_select_topic = widgets.SelectMultiple(
    options=thai_topics,
    description='ประเด็น:',
    value=['System'],
    rows=6
)

output = widgets.Output()

def update_table(change=None):
    with output:
        clear_output()
        selected = list(multi_select_topic.value)
        sub = (
            df_words[df_words['lazada_topics'].isin(selected)]
            .sort_values(by='count', ascending=False)
        )
        if len(sub) > 0:
            display(sub)
        else:
            print("ไม่มีคำสำหรับประเด็นที่เลือก")

multi_select_topic.observe(update_table, names='value')
display(multi_select_topic, output)
update_table()


สร้างพจนานุกรมประเด็นจากการผสานคำสำคัญที่อ้างอิงงานวิจัยและคำที่พบจริงจากข้อมูลรีวิว

In [ ]:
aspect_dict = {

     "System": [
      'กด', 'กดใส่', 'กติกา', 'กรอก', 'กระตุก', 'กลับคืน', 'กลไก', 'กล้อง', 'กากบาท', 'การ์ด',
      'ข้อมูล', 'ครอบคลุม', 'คลิก', 'คลิป', 'ควบคู่', 'คอนเทนต์', 'คัดลอก', 'คำเสิร์ช', 'คีย์เวิร์ด', 'คืบหน้า',
      'ค้น', 'ค้นหา', 'ค้าง', 'จอหลัก', 'จับเวลา', 'จำนวน', 'จำรหัส', 'ชนิด', 'ชั่วคราว', 'ชาญฉลาด',
      'ชำรุด', 'ช่วยเหลือ', 'ซับซ้อน', 'ซัพพอร์ต', 'ดาวน์โหลด', 'ดีเลย์', 'ดีไซน์', 'ตกลง', 'ตรงบัตร', 'ตรงภาพ',
      'ตรวจสอบ', 'ตระกร้า', 'ตะกร้า', 'ถอนตัง', 'ถูกล็อค', 'ทดลอง', 'ทดสอบ', 'ทีม', 'ธุรกรรม', 'บกพร่อง',
      'บล็อก', 'บล็อค', 'บัญชีลาซาด้า', 'บัตร', 'บันทึก', 'บั๊ก', 'บาร์โค้ด', 'บิลทันใจ', 'ประวัติ', 'ประสบการณ์',
      'ประสิทธิภาพ', 'ปรับปรุง', 'ปรับเปลี่ยน', 'ปัญหา', 'ปิดบัญชี', 'ปิดยาก', 'ปุ่ม', 'ปุ่มกากะบาท', 'ปุ่มแคนเซิล', 'ป็อปอัพ',
      'ผลิตภัณฑ์', 'ผันผวน', 'ผิดพลาด', 'พรีเมียม', 'พร้อมเพย์', 'พัง', 'พัฒนา', 'พิจารณา', 'พิสูจน์', 'ฟอนต์',
      'ฟั่งชั่น', 'ฟีด', 'ฟีดแบค', 'ฟีเจอร์', 'ภาษา', 'มาตรการ', 'ยกเลิกบัญชีลาซาด้าวอลเล็ต', 'ย้อนกลับ', 'ย้อนหลัง', 'รวน',
      'รหัส', 'รอ', 'รองรับ', 'ระงับ', 'ระบบ', 'รับรหัส', 'รายงาน', 'รีพอร์ต', 'รีเซ็ท', 'รูปแบบ',
      'รู้จัก', 'ละเอียด', 'ลักษณะ', 'ลาซาด้าวอลเล็ต', 'ลาซาด้าเพย์', 'ลิงก์', 'ลิงค์', 'ลื่น', 'ล็อก', 'ล็อค',
      'ล็อคจอ', 'ล็อคบัญชี', 'ล่ม', 'ล้มเหลว', 'ล๊อคยูส', 'วงเงิน', 'วอลเล็ต', 'วอเล็ต', 'วิธีการ', 'วุ่นวาย',
      'สดุด', 'สถานะ', 'สมัคร', 'สรุปราคา', 'สว่าง', 'สอบถาม', 'สะดวก', 'สับสน', 'สำรอง', 'สเปค',
      'สแกน', 'สแปม', 'หงุดหงิด', 'หน่วง', 'หน้า', 'หน้าเพจ', 'หลักการ', 'อธิบาย', 'อนุมัติ', 'ออกแบบ',
      'ออนไลน์', 'ออโต้', 'อัปเดต', 'อัปเดตการ', 'อัปเดทล่าสุด', 'อัพ', 'อัพเดต', 'อัพเดตรอ', 'อัพเดทรีวิว', 'อัพเดทเวอร์ชั่น',
      'อัพเดทแอพ', 'อัพโหลด', 'อัฟเกรด', 'อัฟโหลด', 'อินเตอร์เน็ต', 'อินเทอร์เน็ต', 'อืด', 'อ่าน', 'เกณฑ์', 'เข้ารหัส',
      'เครื่องหมาย', 'เคลม', 'เช็ก', 'เดบิต', 'เด้ง', 'เทคนิค', 'เทสระบบ', 'เท็จจริง', 'เนื้อหา', 'เป็นเวอร์ชั่น',
      'เม้นท์', 'เรียลไทม์', 'เลขรหัส', 'เวป', 'เวอร์', 'เวอร์ชั่น', 'เว็บ', 'เว็บบราวเซอร์', 'เว็บลาซาด้า', 'เว็ป',
      'เว็ปกด', 'เสถียร', 'เอกสาร', 'เอพเสถียรดี', 'เอไอ', 'เอ๋อ', 'แก้', 'แชต', 'แตกต่าง', 'แตะ',
      'แม่นยำ', 'แรม', 'แอด', 'แอดโฆษณา', 'แอปกระจอก', 'แอปกาก', 'แอปค้าง', 'แอปชอปบี้', 'แอปช็อปปิ้ง', 'แอปช็อปสินค้า',
      'แอปพัฒนาการ', 'แอปรวน', 'แอปลื่น', 'แอปห่วย', 'แอปออนไลน์', 'แอปเนื้อหา', 'แอพดี', 'แอพเชย', 'แอพเสถียร', 'แอพเสถียรดี',
      'แอพโอเค', 'แอพไวรัส', 'แอฟ', 'แอฟช้า', 'แอฟลาซาด้า', 'แอ็ป', 'แฮ็ก', 'โฆษณา', 'โฆษา', 'โชว์','แอพ',
      'โปรแกรม', 'โพสต์', 'โหมด', 'โหมดวิดีโอหาย', 'โหลด', 'โหลดลาซาด้า', 'โอน', 'ใส่รหัส', 'ไอคอน','เติม',

    ],

     "Logistics": [
      'กล่อง', 'การขนส่ง', 'ขนส่ง', 'ขนส่งทาง', 'ขนส่งพอ', 'ขนส่งอะรัย', 'ขนส่งไว', 'ขับไว', 'คลัง', 'คัดแยก',
      'จอง', 'ฉีกขาด', 'ดีขนส่ง', 'ตกค้าง', 'ตรวจ', 'ตรวจเช็ค', 'ติดขัด', 'ติดค้าง', 'ติดตาม', 'ต่อเนื่อง',
      'ถ่าย', 'ทำขนส่ง', 'นัด', 'บริษัทขนส่ง', 'ปกแพ็ค', 'ประทับใจขนส่ง', 'ปริมาณ', 'ปลาย', 'ปัญหา', 'พัสดุ',
      'พัสดุกด', 'พัสดุดี', 'พัสดุมัน', 'พัสดุหาย', 'ฟรีส่ง', 'ภาพ', 'มากพัสดุ', 'ยกเลิก', 'รับคืน',
      'ราคาขนส่ง', 'ล้าช้า', 'ล้าหลัง', 'วันหาย', 'ว่างส่ง', 'ศุลกากร', 'ศูนย์กระจาย', 'สินค้าเคลมยาก', 'สิ่งของ', 'สุภาพ',
      'ส่งกอ้', 'ส่งง', 'ส่งท้าย', 'ส่งพอประมาณ', 'ส่งพอใช้', 'ส่งมอบ', 'หมายเลขพัสดุ', 'หลุด', 'หล่น', 'หอบหิ้ว',
      'หาย', 'ห่อ', 'ออเดอร์', 'อัพเดตสถานะ', 'เคลมสินค้า', 'เค้าเตอร์เซอร์วิสหาย', 'เจอ', 'เร่งรีบ', 'เลขพัสดุ', 'เลื่อน',
      'เลื่อนส่ง', 'เสียหาย', 'แตก', 'แผนที่', 'แพคสินค้า', 'แพ็ค', 'แพ็คพัสดุ', 'แพ้ค', 'แสกนสินค้า', 'ไปรษณีย์',
      'ไปษณีย์', 'ไลน์แมน', 'ไวแพ็คสินค้า'

    ],

     "Company": [
      'การค้า', 'การันตี', 'กำกับ', 'ก่อกวน', 'ก้บริการ', 'ขอร้องเรียน', 'ขโมย', 'คอมเพลน', 'คอมเม้นต์', 'คอมเม้นท์',
      'คอลเซนเตอร์', 'คืนตัง', 'คุ้มครอง', 'คู่แข่ง', 'ชดเชย', 'ซาด้า', 'ดำเนินการ', 'ดีงาม', 'ดูแลรักษา', 'ตอบ',
      'ตอบสนอง', 'ตอบแทน', 'ตำหนิ', 'ติดต่อ', 'ถอน', 'ทดแทน', 'ทำไมลาซาด้า', 'ทีม', 'ธุรกิจ', 'นโยบาย',
      'นโยบายลาซาด้า', 'บริการ', 'บัญชี', 'ประจำลาซาด้า', 'ประชาสัมพันธ์', 'ประทับใจ', 'ประพฤติ', 'ประสงค์', 'ประเด็น', 'ประเทศ',
      'ปรึกษา', 'ปลาซาด้า', 'ปิดกั้น', 'ผิดเจ้าของ', 'พนักงาน', 'พรีเซนเตอร์', 'พรีเมียม', 'พฤติกรรม', 'พฤติการณ์', 'พาร์ทเนอร์',
      'พิจารณา', 'พิม', 'พี่ลาซ', 'พูดจา', 'ฟ้อง', 'มั่นใจ', 'มาตรฐาน', 'มาตรา', 'มาร์เก็ตติ้ง', 'มาลาซาด้า',
      'มิชลิน', 'ยินยอม', 'ยุติธรรม', 'รบกวน', 'ระดับ', 'รับผิดชอบ', 'รับฟัง', 'รั่วไหล', 'ราคา', 'รางวัล',
      'ราชาดามั่นใจ', 'ราชาด้า', 'ราสาดารฺ์', 'รีเฟรช', 'ลาชด้า', 'ลาชาดา', 'ลาซรีวอร์ด', 'ลาซาก้า', 'ลาซาดาวอเล็ท', 'ลาซาด้า',
      'ลาซาด้าครวรับผิดชอบ', 'ลาซาด้าละ', 'ลาซาด้าวอลเล็ท', 'ลาซาด้าเพย์เลเทอ', 'ลาซาด้าเลตเตอร์', 'ลาซาด้าเลสเตอร์', 'ลาซาด้าเฮ็ก', 'ลาซ้าดา', 'ลูกค้า', 'ลูกค้าลาซาด้า',
      'วงจร', 'ศาด้า', 'ศึกษา', 'ศูนย์ใหญ่', 'สนับสนุน', 'สนใจ', 'สปอนเซอร์', 'สมาชิก', 'สัมพันธ์', 'สิทธิ', 'ใส่ใจ',
      'สินค้าลาซาด้า', 'หลอกลวง', 'หลักฐาน', 'ออกแบบ', 'ออร์เดอร์', 'อัตรา', 'อีลาซาด้า', 'เข้มงวด', 'เข้าลาชาด้าวอเลต', 'เคร่งครัด',
      'เคลียร์', 'เคารพ', 'เงินตรา', 'เงินลาซาด้าว', 'เงียบหาย', 'เงื่อนไข', 'เจ้าหน้า', 'เจ้าหน้าที่', 'เจ้าหน้าที่นาน', 
      'เถียง', 'เบอร์', 'เบอร์พนง', 'เบิร์ดเดย์ลาซาด้า', 'เป็นธรรม', 'เป้าหมาย', 'เพิกเฉย', 'เรียกร้อง', 'เว็บไซต์', 'เสนอ',
      'เสี่ยง', 'เหมาะสมบริการ', 'เอกสาร', 'เอดมิน', 'เอาเปรียบ', 'แนะนำ', 'แบน', 'แบนเนอร์', 'แบรนเนอร์', 'แบล็คลิสลาซาด้า',
      'แผนก', 'แพลตฟอร์ม', 'แฟล็กร้าน', 'แอดมิน', 'แอปพลิเคชัน', 'แอพพลิเคชั่น', 'โกง', 'โทร', 'โทรจิก', 'โบนัส',


    ],


     "Store": [
      'กระทบ', 'กระเป๋า', 'กวนใจ', 'กางเกง', 'กุ้ง', 'ขนาด', 'ของขวัญ', 'ขัดเงา', 'ขาย', 'ข้าว',
      'คัดสินค้า', 'คัดเลือก', 'คิวอาร์โค้ด', 'คืน', 'คุณภาพ', 'คุณภาพบาง', 'คุ้มช๊อปออนไลน์', 'คูณภาพ', 'คูปองลูกค้า', 'ค้าซื่อสัตย์',
      'จักรยาน', 'จัดซื้อ', 'จับจ่าย', 'จำหน่าย', 'ชอปง่าย', 'ชอปปิ้ง', 'ชอ้ป', 'ชำระสินค้า', 'ชื่นชอบ', 'ชื่อ',
      'ช็อบง่าย', 'ช็อปดี', 'ช็อปปิ้ง', 'ช็อปฯ', 'ช้อป', 'ช้อปจริง', 'ช้อปปิ้ง', 'ช้อปสินค้า', 'ช๊อปดี', 'ซื้อขาย',
      'ซื้อขายแลกเปลี่ยน', 'ซื้อหา', 'ดองออเดอร์', 'ดาว', 'ดำเนิน', 'ดีเยี่ยม', 'ดีเลิศ', 'ดีไซน์', 'ดุ', 'ตรงปก',
      'ตลาดเปลือง', 'ตำหนิขาย', 'ติดใจ', 'ต้นทุน', 'ต้ม', 'ถูกใจจริง', 'ทนทาน', 'ท็อปช้อย', 'ท้องตลาด',
      'นิยาย', 'น้ำหนัก', 'บรรจุภัณฑ์', 'บุบ', 'บูทประชาสัมพันธ์', 'บ้าน', 'ปกราคา', 'ประกัน', 'ประทับใจ', 'ประมูล',
      'ประหยัด', 'ปลีก', 'ปัก', 'ปั่น', 'ปากกา', 'ปากต่อปาก', 'ผลลัพธ์', 'ผลิต', 'ผลิตภัณฑ์สินค้า', 'ผ่อนจ่าย',
      'ผ่อนร่วม', 'ผ้า', 'พนัก', 'พรีออเดอร์', 'พรีเซ็นเตอร์', 'พอใจ', 'พิกัด', 'พิเศษ', 'มือถือ', 'มือสอง',
      'มุ้ง', 'ยอด', 'ยี่ห่อ', 'ยี่ห้อ', 'ยุ่งยากราคา', 'รสชาติ', 'รองเท้า', 'รอย', 'รับออเดอร์', 'ราคา',
      'ราคากัพอดี', 'ราคาสมเหตุสมผล', 'ราคาเพียบ', 'ราซาด้ามอล์', 'รายละเอียด', 'รีวิว', 'รูป', 'รูปภาพ', 'ร่องรอย', 'ร้สินค้า',
      'ร้านช๊อป', 'ร้านน', 'ร้านเค่เพ็ค', 'ลาคา', 'ลาซ้าด้า', 'วงเงินดี', 'วัสดุ', 'วัสดุดี', 'สต็อก', 'สบายใจ',
      'สบู่', 'สภาพ', 'สมราคา', 'สร้าง', 'สวยงาม', 'สะดวกสบายดีจิง', 'สะดวด', 'สั่งการ', 'สั่งงาย', 'สั่งชื้อ',
      'สั่งสิงค้า', 'สาย', 'สิงค้า', 'สิน', 'สินค้า', 'สินค้ารคา', 'สินค้าราคา', 'สินค้่', 'สินดี', 'สินต้า',
      'สิ่งของ', 'สูญหาย', 'สเปคดี', 'สแกน', 'หนังสือ', 'หลากหลาย', 'หวาน', 'หัก', 'ห้าง', 'ห้างร้าน',
      'อะไหล่', 'อาหาร', 'อุดหนุน', 'เกรด', 'เครื่อง', 'เจ้าของ', 'เชื้อ', 'เต่สินค้า', 'เนื้อ', 'เป็นชิ้นเป็นอัน',
      'เรียบร้อย', 'เละ', 'เลือก', 'เสริม', 'เสียหลอกลวง', 'เสียหาย', 'เสื้อผ้า', 'เหมาะสม', 'แกง',
      'แข็งแรง', 'แคป', 'แชร์', 'แตก', 'แบรนด์เนม', 'แพง', 'แพ็กมา', 'แพ็ค', 'แพ็คเกจ', 'แฟชั่น',
      'แม่ค้า', 'แหล่ง', 'แอพลาซาด้า', 'โปรจัด', 'ไซส์', 'ไลฟ์', 'ไอ่สินค้า'


    ],


     "Promotion": [
      'กระหน่ำ', 'กั้กโค้ด', 'กิจกรรม', 'กินโค้ด', 'ของขวัญเลือก', 'คะแนน', 'คุปอง', 'คูปอง', 'คูปองกี่', 'คูปองคืน',
      'คูปองซื้อ', 'คูปองดี', 'คูปองยาก', 'คูปองเงิน', 'คูปองเจก', 'คูปองเหรียน', 'คูปองโสน', 'คูปองใช้', 'จริงโค้ด', 'ฉ่ำโค้ด',
      'ชำระ', 'ดีล', 'ดีลสินค้า', 'บางโค้ด', 'บาท', 'ประหยัด', 'พอยต์', 'พิเศษ', 'ฟรี', 'ฟรีสิทธิ์',
      'ฟรีเยี่ยม', 'มีโค้ดเต่', 'มีโค๊ต', 'มูลค่า', 'ย่อมเยาว์', 'รอกดคูปอง', 'รอโค้ด', 'รักษาสิทธิ์', 'ราคาเบาลด', 'ลด',
      'ลดคุ้ม', 'ลดยากอ่ะ', 'ลดลา', 'ลดเยอะ', 'ลดโค้ด', 'วีไอพี', 'ศูนย์บาท', 'สุ่มแจก', 'ส่งโค้ด', 'ส่วฟรี',
      'เกม', 'เครดิต', 'เควสต์', 'เต็ม', 'เผยแพร่', 'เฟรสเซล', 'เลข', 'เละโค้ด', 'เหรียญ', 'แคมเปญ',
      'แจกเหรียญ', 'แจกแถม', 'แตกโค้ด', 'แถม', 'แบนด์', 'แบนโค้ด', 'แฟลชเซล', 'แย่งโค้ด', 'แลก', 'แลกคูปอง',
      'แลกแจก', 'แลกโค้ด', 'โคด', 'โค้ด', 'โค้ดทิพย์', 'โค้ดหาย', 'โค้ดแฟลชราย', 'โค้ดโหดเว่อร์', 'โค้ต', 'โค๊ด',
      'โค๊ต', 'โปร', 'โปรดี', 'โปรต่อ', 'โปรที', 'โปรมีคุปอง', 'โปรลด', 'โปรลูกค้า', 'โปรสะสม', 'โปรส่ง',
      'โปรส่วน', 'โปรเพียบ', 'โปรเยอะ', 'โปรเลย', 'โปรแรง', 'โปรโคตรแรง', 'โปรโมชัน', 'โปรโมชั่นพิเศษดี', 'โปรโมชั่นวัน', 'โปรโมทชั่นแทรก',
      'โปรโม่ชั่น', 'โอน', 'ใช้จ่าย'
        ]
    }


ตรวจสอบคำใน aspect_dict ว่ามีจริงในข้อมูล

In [ ]:
# 1) เตรียม tokens จริงจาก df_content
# explode ให้เป็นคำเดี่ยว
all_tokens_series = df_content["tokens"].explode()

# นับจำนวนจริงของแต่ละคำ
token_counter = Counter(all_tokens_series)

# เก็บเป็น set สำหรับเช็คว่ามีคำหรือไม่
tokens_set = set(all_tokens_series)

print(" จำนวน token ไม่ซ้ำทั้งหมดในรีวิว :", len(tokens_set))


# 2) ฟังก์ชันตรวจ:
#    - คำซ้ำใน aspect
#    - คำที่ไม่มีอยู่ในรีวิว
#    - คำที่พบบ่อยจริงในข้อมูล
def check_aspect_words(aspects, tokens_set, token_counter):
    results = {}

    for aspect, words in aspects.items():

        # ลบคำซ้ำใน list เดิม
        unique_words = list(dict.fromkeys(words))

        # --- คำซ้ำที่เคยมีใน original list ---
        word_count = Counter(words)
        duplicates = [w for w, c in word_count.items() if c > 1]

        # --- คำที่ไม่มีในรีวิวเลย ---
        missing = [w for w in unique_words if w not in tokens_set]

        # --- คำที่พบจริง + จำนวนที่พบ ---
        found = {
            w: token_counter[w]
            for w in unique_words
            if w in tokens_set
        }

        results[aspect] = {
            "duplicate_words": duplicates,
            "missing_in_tokens": missing,
            "found_in_data": found
        }

    return results

# 3) เรียกฟังก์ชัน
results = check_aspect_words(aspect_dict, tokens_set, token_counter)


# แสดงผลแบบ list เรียงสวย ๆ
for aspect, res in results.items():
    print("\n" + "="*50)
    print(f" ASPECT : {aspect}")

    print("\nคำที่ไม่พบในข้อมูลจริง:")
    if res["missing_in_tokens"]:
        # เรียงคำตามตัวอักษร + ตัดซ้ำ
        sorted_missing = sorted(list(set(res["missing_in_tokens"])))
        print(sorted_missing)
    else:
        print("ทุกคำพบในข้อมูล")


    print("\nคำที่พบในข้อมูลจริง:")
    if res["found_in_data"]:
        # เรียงคำตามตัวอักษรเท่านั้น ไม่โชว์จำนวน
        sorted_found = sorted(list(res["found_in_data"].keys()))
        print(sorted_found)
    else:
        print("ไม่พบคำใดในข้อมูล")



## ส่วนที่3 การจำแนกประเด็นด้วย Machine Learning ( Naive Bayes & RF)

ส่วนนี้มีเป้าหมายเพื่อจำแนกประเด็นที่ผู้ใช้พูดถึงในรีวิว
โดยเป็นการจำแนกแบบ Multi-label ระดับประโยค
เนื่องจากไม่มี label จริงจากข้อมูล
จึงใช้วิธี Keyword-based labeling เพื่อสร้าง label
แล้วนำข้อมูลดังกล่าวไปฝึกและเปรียบเทียบโมเดล Naive Bayes และ Random Forest

In [ ]:
import numpy as np
import joblib
import re
from pythainlp.tokenize import sent_tokenize
from pythainlp.util import normalize
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# ใช้ชุดข้อมูลเต็มที่แยกไว้แล้ว (df_full)
df = df_full.copy()

print("ข้อมูลทั้งหมดในไฟล์ (ชุดข้อมูลเต็ม)")
print("จำนวนแถว:", len(df))
print("จำนวนคอลัมน์:", len(df.columns))
display(df.head(10))


In [ ]:
df_content = df[["content", "year", "month"]].copy()

print("\n ข้อมูลที่ใช้สำหรับการวิเคราะห์ (df_content)")
print("จำนวนแถว:", len(df_content))
display(df_content.head(10))


ทำความสะอาดข้อความ (Text Cleaning)

In [ ]:
df_content["clean_text"] = df_content["content"].apply(clean_thai)
df_content["tokens"] = df_content["clean_text"].apply(tokenize_thai)

print("ตัวอย่าง token:")
display(df_content[["content", "tokens"]].head(5))


สร้าง label จากคำสำคัญ (Keyword-based labeling)

* กำหนด หมวดประเด็น (Aspect) ที่ต้องการวิเคราะห์
* เพิ่มหมวด Other เพื่อรองรับรีวิวที่ไม่เข้าเกณฑ์ใดเลย
* รับ input เป็น tokens ของข้อความ
* ตรวจสอบว่า token ใดตรงกับคำใน aspect_dict ถ้าพบ → ใส่หมวดนั้น  ถ้าไม่พบเลย → จัดเป็น "Other"

In [ ]:
aspect_cols = ["System", "Logistics", "Company", "Store", "Promotion", "Other"]

def classify_aspects(tokens):
    aspects = []
    for asp, words in aspect_dict.items():
        if any(w in tokens for w in words):
            aspects.append(asp)
    if not aspects:
        aspects = ["Other"]
    return aspects

วิเคราะห์จำนวนรีวิวในแต่ละประเด็น 
* นำฟังก์ชันไปใช้กับ รีวิวทั้งก้อน เพื่อตรวจสอบว่าหมวดไหนมีจำนวนรีวิวเท่าไหร่ 



In [ ]:
# 2) ใส่ผลลัพธ์ aspect ลงใน df_content
df_content["aspects"] = df_content["tokens"].apply(classify_aspects)

# นับจำนวนรีวิวของแต่ละหมวด (รวม Other)
aspect_review_count = Counter()
for asp_list in df_content["aspects"]:
    for a in asp_list:
        aspect_review_count[a] += 1

df_aspect_count = (
    pd.DataFrame.from_dict(aspect_review_count, orient='index', columns=['review_count'])
      .sort_values(by="review_count", ascending=False)
)
print("จำนวนรีวิวในแต่ละหมวด (รวมหมวดอื่นๆ):")
display(df_aspect_count)


แยกรีวิวเป็นระดับประโยค
* รีวิว 1 อัน อาจพูดหลายประเด็น การจำแนกระดับประโยคทำให้แม่นยำกว่า
* สร้าง dataset ใหม่ในระดับ sentence เพื่อเพิ่มความแม่นยำในการจำแนก aspect
* คลีนประโยคใหม่
* ลบประโยคที่ว่างหลังคลีน


In [ ]:
# สร้าง df_sentences จากประโยค
def split_raw_sentences(text):
    sents = sent_tokenize(str(text))
    return [s.strip() for s in sents if s.strip() != ""]

df_content["sentences_raw"] = df_content["content"].apply(split_raw_sentences)
df_sentences = df_content.explode("sentences_raw").reset_index(drop=True)
df_sentences = df_sentences.rename(columns={
    "content": "review_text",
    "sentences_raw": "sentence_raw"
})

# Clean ประโยค
def clean_sentence(text):
    text = normalize(text)
    text = re.sub(r"[^\u0E00-\u0E7F a-zA-Z0-9]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_sentences["sentence_clean"] = df_sentences["sentence_raw"].apply(clean_sentence)

#ลบประโยคที่ว่าง หลัง clean (จุดสำคัญ)
df_sentences = df_sentences.dropna(subset=["sentence_clean"])
df_sentences = df_sentences[df_sentences["sentence_clean"].str.strip() != ""]


Tokenize ประโยค (แปลงข้อความเป็นคำ)
* นำข้อความที่ผ่านการ clean แล้วในแต่ละประโยคแยกออกเป็น รายการคำ (tokens)
* ผลลัพธ์ของแต่ละแถวจะเป็น list ของคำ
* ตรวจว่าประโยคกล่าวถึง ประเด็น (aspect) ใดบ้าง

In [ ]:
# Tokenize และ Aspect classification
df_sentences["tokens"] = df_sentences["sentence_clean"].apply(tokenize_thai)
df_sentences["aspects"] = df_sentences["tokens"].apply(classify_aspects)

# ใช้ MultiLabelBinarizer แทน loop สร้าง one-hot
mlb = MultiLabelBinarizer(classes=aspect_cols)
y = pd.DataFrame(mlb.fit_transform(df_sentences["aspects"]),
                 columns=mlb.classes_, index=df_sentences.index)

# สร้าง columns ของแต่ละ aspect ใน df_sentences สำหรับ display
for col in aspect_cols:
    df_sentences[col] = y[col]

# Display ตัวอย่าง
cols_display = ["review_text", "sentence_raw", "sentence_clean"] + aspect_cols
df_display = df_sentences[cols_display]

def highlight_ones(val):
    return "background-color: lightgreen" if val == 1 else ""

display(df_display.style.applymap(highlight_ones, subset=aspect_cols))
print("✓ df_sentences สร้างสำเร็จ จำนวนแถว:", len(df_sentences))

รวมคำกลับเป็นข้อความ (เตรียมข้อมูลเข้าโมเดล)
* นำ tokens ที่อยู่ในรูปแบบ list ของคำ มาต่อกลับเป็นข้อความเดียว
* คั่นแต่ละคำด้วยเว้นวรรค เพื่อให้โมเดลอย่าง TF-IDF สามารถอ่านและประมวลผลได้


เตรียมข้อมูล X (ข้อความสำหรับเทรนโมเดล)
* กำหนดคอลัมน์ข้อความที่รวมคำแล้วเป็นตัวแปร X



In [ ]:
df_sentences["tokens_join"] = df_sentences["tokens"].apply(
    lambda toks: " ".join(toks)
)

X = df_sentences["tokens_join"].astype(str)

print("ขนาดข้อมูล X:", X.shape)
print("ขนาด label y:", y.shape)
print("\nจำนวนข้อมูลของแต่ละหมวด:")
display(y.sum())

# แบ่งข้อมูลเป็นชุด Train และ Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


ขั้นตอนเตรียมข้อมูลให้โมเดล Random Forest หรือ Naive Bayes ใช้ได้ โดยแปลงข้อความเป็นตัวเลข TF-IDF

แปลงข้อความเป็นตัวเลขด้วย TF-IDF
* TF-IDF จะดูว่าคำไหนสำคัญในประโยคนั้น และไม่ใช่คำที่โผล่บ่อยเกินไปในทุกประโยค ทำให้โมเดลโฟกัสคำที่ช่วยแยกประเด็นได้จริง


กำหนด min_df=5 เลือกใช้เฉพาะคำที่ปรากฏอย่างน้อย 5 ประโยค
* ตัดคำที่เจอน้อยมาก ซึ่งมักเป็นคำพิมพ์ผิดหรือไม่สำคัญ
* ช่วยลด noise และทำให้โมเดลเสถียรมากขึ้น

กำหนด ngram_range=(1,2)
* ใช้ทั้งคำเดี่ยว (unigram) และคำคู่ (bigram)
* ช่วยให้โมเดลเข้าใจความหมายเป็นวลี เช่น จาก "คิวอาร์ ผิดพลาด ปรับปรุง" เป็น คิวอาร์ ผิดพลาด ทำให้เห็นประเด็นชัดยิ่งขึ้น

หลังจากการแปลงด้วย TF-IDF ข้อมูลจะอยู่ในรูปแบบเชิงตัวเลข ซึ่งเป็นรูปแบบที่โมเดล Machine Learning สามารถประมวลผลได้โดยตรง

In [ ]:
# TF-IDF Vectorizer
# -------------------------------
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    min_df=5,
    ngram_range=(1,2)
)
#เรียนรู้ vocabulary จาก ชุด train และแปลงเป็น TF-IDF
X_train_tfidf = vectorizer.fit_transform(X_train)
#แปลง ชุด test ด้วย vocabulary เดียวกับ train
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape (train):", X_train_tfidf.shape)
print("จำนวนคำทั้งหมดใน vocabulary:", len(vectorizer.vocabulary_))
print("ตัวอย่าง X (tokens_join):")
print(X.head(10))
print("ตัวอย่าง y (one-hot per aspect):")
print(y.head(10))


เอาข้อมูลข้อความที่แปลงเป็น TF-IDF แล้ว ไปฝึกโมเดล Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score


# Train Naive Bayes (One-vs-Rest)
nb_model = OneVsRestClassifier(MultinomialNB(), n_jobs=-1)

print("เริ่มเทรน Naive Bayes ...")
nb_model.fit(X_train_tfidf, y_train)
print("เทรนเสร็จแล้ว")

# Predict และ Evaluate
y_pred_nb = nb_model.predict(X_test_tfidf)

print("===== รายงานผล Naive Bayes (multi-label) =====\n")
print(classification_report(y_test, y_pred_nb, digits=4, target_names=aspect_cols, zero_division=0))

acc_per_label_nb = {asp: accuracy_score(y_test[asp], y_pred_nb[:, i])
                    for i, asp in enumerate(aspect_cols)}
print("\nAccuracy ต่อแต่ละหมวด (NB):")
for asp, acc in acc_per_label_nb.items():
    print(f"{asp}: {acc:.4f}")

macro_acc_nb = np.mean(list(acc_per_label_nb.values()))
micro_acc_nb = accuracy_score(y_test.values.flatten(), y_pred_nb.flatten())
print(f"\nAccuracy เฉลี่ยแบบ Macro (NB): {macro_acc_nb:.4f}")
print(f"Accuracy แบบ Micro (NB): {micro_acc_nb:.4f}")



In [ ]:
# Train One-vs-Rest Random Forest
# สร้างโมเดล Random Forest พื้นฐาน
# - n_estimators=50 : ใช้ต้นไม้ตัดสินใจ 50 ต้น
# - class_weight="balanced" : ช่วยแก้ปัญหาข้อมูลแต่ละหมวดไม่สมดุล
# - random_state=42 : ทำให้รันซ้ำแล้วได้ผลลัพธ์เดิม
# - n_jobs=-1 : ใช้ CPU ทุกคอร์เพื่อให้ประมวลผลเร็วขึ้น
base_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# ใช้ One-vs-Rest เพื่อให้ Random Forest
# สามารถทำนายได้มากกว่าหนึ่งประเด็นในหนึ่งประโยค
model = OneVsRestClassifier(base_model, n_jobs=-1)

# เริ่มเทรนโมเดลด้วยข้อมูล train ที่แปลงเป็น TF-IDF แล้ว
print("เริ่มเทรนโมเดล Random Forest ...")
model.fit(X_train_tfidf, y_train)
print("เทรนเสร็จแล้ว")

# ทำนายผลบนชุดข้อมูล test
y_pred = model.predict(X_test_tfidf)

# แสดงรายงานผลการจำแนกแบบ multi-label
# แสดง precision, recall และ f1-score แยกตามแต่ละประเด็น
print("===== รายงานผลรวมทุกหมวด (multi-label) =====\n")
print(classification_report(y_test, y_pred, digits=4, target_names=aspect_cols, zero_division=0))

# คำนวณ Accuracy แยกตามแต่ละหมวด
acc_per_label = {asp: accuracy_score(y_test[asp], y_pred[:, i])
                 for i, asp in enumerate(aspect_cols)}
print("\nAccuracy ต่อแต่ละหมวด:")
for asp, acc in acc_per_label.items():
    print(f"{asp}: {acc:.4f}")


macro_acc = np.mean(list(acc_per_label.values()))
micro_acc = accuracy_score(y_test.values.flatten(), y_pred.flatten())
print(f"\nAccuracy เฉลี่ยแบบ Macro: {macro_acc:.4f}")
print(f"Accuracy แบบ Micro (รวมทุก label): {micro_acc:.4f}")


In [ ]:
# สร้าง DataFrame สำหรับเอาไว้ดูและเปรียบเทียบผลทำนาย
# ดึงเฉพาะแถวที่เป็นชุด test มาใช้
df_pred_compare = df_sentences.loc[y_test.index].copy()

# เก็บ label จริงของแต่ละประโยค (จากlabel ที่เตรียมไว้)
df_pred_compare["true_aspects"] = df_pred_compare["aspects"]

# แปลงผลทำนายจาก Naive Bayes
# จากค่า 0/1 ให้เป็นรายชื่อหมวด (aspect) ที่โมเดลทำนายว่าเจอ
df_pred_compare["pred_nb"] = [
    [asp for i, asp in enumerate(aspect_cols) if y_pred_nb[idx, i] == 1]
    for idx in range(len(y_pred_nb))
]

# แปลงผลทำนายจาก Random Forest
# ทำแบบเดียวกับ Naive Bayes แต่เป็นผลจากโมเดล RF
df_pred_compare["pred_rf"] = [
    [asp for i, asp in enumerate(aspect_cols) if y_pred[idx, i] == 1]
    for idx in range(len(y_pred))
]

# เลือกคอลัมน์ที่จะแสดง
cols_show = [
    "sentence_raw",  # ประโยคต้นฉบับ
    "true_aspects",  # หมวดจริง
    "pred_nb",       # ผลทำนายจาก Naive Bayes
    "pred_rf"        # ผลทำนายจาก Random Forest
]

display(
    df_pred_compare[cols_show].head(5)
)


จัดเก็บผลการทำนาย Aspect จากโมเดล Random Forest

In [ ]:
# 1) สร้าง TF-IDF จากทั้งชุด
X_all_tfidf = vectorizer.fit_transform(X)

# 2) เทรน RF กับทั้งชุด
model_full = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    n_jobs=-1
)
model_full.fit(X_all_tfidf, y)

# 3) ทำนายผลทั้งชุด
y_pred_all = model_full.predict(X_all_tfidf)

# 4) เพิ่มคอลัมน์ผลทำนาย RF ลงใน df_sentences
for i, asp in enumerate(aspect_cols):
    df_sentences[f"RF_{asp}"] = y_pred_all[:, i]

# 5) เซฟไฟล์
cols_to_save = [
    "year", "month",           
    "review_text",
    "sentence_raw",
    "sentence_clean",
    "tokens_join",
    "aspects"
] + [f"RF_{asp}" for asp in aspect_cols]

df_sentences[cols_to_save].to_csv(
    "df_sentences_for_use.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ บันทึกไฟล์ df_sentences_for_use.csv เรียบร้อย")


In [ ]:
# -------------------------------
# Train NB กับทั้งชุด
# -------------------------------
nb_full = OneVsRestClassifier(MultinomialNB(), n_jobs=-1)
nb_full.fit(X_all_tfidf, y)

# ทำนายทั้งชุด
y_pred_all_nb = nb_full.predict(X_all_tfidf)

# เพิ่มคอลัมน์ผลทำนาย NB ลงใน df_sentences
for i, asp in enumerate(aspect_cols):
    df_sentences[f"NB_{asp}"] = y_pred_all_nb[:, i]

# -------------------------------
# เซฟไฟล์ NB แยกไฟล์
# -------------------------------
cols_to_save_nb = [
    "year", "month",
    "review_text",
    "sentence_raw",
    "sentence_clean",
    "tokens_join",
] + [f"NB_{asp}" for asp in aspect_cols]

df_sentences[cols_to_save_nb].to_csv(
    "df_sentences_nb_only.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ บันทึกไฟล์ df_sentences_nb_only.csv เรียบร้อย")


บันทึกโมเดล

In [ ]:
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(nb_model, "nb_model.joblib")
joblib.dump(model, "rf_model.joblib")

## ส่วนที่ 4 การวิเคราะห์ความคิดเห็นเชิงบวก–เชิงลบ–เป็นกลาง

โหลดข้อมูลและเตรียม DataFrame

In [ ]:
import pandas as pd

df_sentences = pd.read_csv(
    "df_sentences_for_use.csv",
    encoding="utf-8-sig"
)

print("โหลดไฟล์สำเร็จ")
print("จำนวนแถว (ก่อนลบ):", df_sentences.shape[0])
print("จำนวนคอลัมน์:", df_sentences.shape[1])

rows_before = df_sentences.shape[0]
df_sentences = df_sentences.dropna(subset=["sentence_clean"])
rows_after = df_sentences.shape[0]

df_sentences.head()


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoConfig

โหลดโมเดล WangchanBERTa สำหรับ Sentiment Analysis
* ใช้ WangchanBERTa ที่ถูก fine-tune สำหรับวิเคราะห์อารมณ์ (positive/negative/neutral)
* ทำงานระดับ sentence-level

In [ ]:
#กำหนดชื่อโมเดลที่ต้องการโหลดจาก Hugging Face 
#ซึ่งเป็นโมเดล WangchanBERTa ที่ถูก fine-tune สำหรับงานวิเคราะห์อารมณ์ 
MODEL_NAME = "poom-sci/WangchanBERTa-finetuned-sentiment"

#โหลด tokenizer ที่ใช้ตัดและแปลงข้อความภาษาไทยให้เป็นรูปแบบที่โมเดลสามารถรับเข้าไปประมวลผลได้
# โดยใช้ tokenizer ตัวเดียวกับที่ใช้ฝึกโมเดล 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#โมเดลสำหรับงานจำแนกประเภทข้อความ (Sequence Classification)
# ซึ่งในที่นี้ใช้สำหรับทำนาย sentiment ของข้อความ
sentiment_model  = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

#โหลด configuration ของโมเดล ซึ่งภายในจะเก็บข้อมูลสำคัญ
# เช่น จำนวนคลาส และ mapping ระหว่างค่าดัชนีของโมเดลกับชื่อ label
config = AutoConfig.from_pretrained(MODEL_NAME)

#ดึง mapping จากค่าดัชนีที่โมเดลทำนาย
id2label = config.id2label
print("Model id2label (mapping จริง):", id2label)

เลือกอุปกรณ์ในการรันโมเดล (CPU หรือ GPU) ย้ายโมเดลไปยังอุปกรณ์นั้น และตั้งค่าโมเดลให้อยู่ในโหมดทำนาย (evaluation mode) เพื่อให้ผลลัพธ์มีความถูกต้องและคงที่

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sentiment_model.to(device)
sentiment_model.eval()  # โหมด evaluation (ปิด dropout, batchnorm ไม่เทรน)
print("Using device:", device)


ฟังก์ชันทำนาย Sentiment แบบ batch
* รับรายการประโยคเป็น input
* ใช้ tokenizer แปลงประโยคเป็น input ของโมเดล
* คืนค่าเป็น list ของ sentiment (pos, neg, neu)

In [ ]:
# ฟังก์ชันทำนาย sentiment แบบ batch
def sentiment_batch_predict(text_list, batch_size=64, max_length=256):
    labels_list = []
    total = len(text_list)

    print("เริ่มทำนายทั้งหมด =", total, "ประโยค")

    for i in range(0, total, batch_size):
        batch = text_list[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = sentiment_model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        preds = preds.cpu().tolist()
        labels = [id2label[p] for p in preds]
        labels_list.extend(labels)

        print(f"Processed {min(i + batch_size, total)}/{total}", end="\r")

    print("\nทำนายเสร็จแล้ว!")
    return labels_list


เตรียมข้อความสำหรับโมเดล
* ใช้คอลัมน์ sentence_clean
* แปลงเป็น list ของ string เพื่อส่งเข้าโมเดล

In [ ]:
# 3) เตรียมข้อความ (ใช้ sentence_clean)
df_sentences = df_sentences.reset_index(drop=True)

text_list = (
    df_sentences["sentence_clean"]
    .fillna("")
    .astype(str)
    .tolist()
)

ทำนาย sentiment และเก็บผลลัพธ์
* เพิ่มคอลัมน์ใหม่ sentiment_pred
* เก็บผลเป็น pos/neg/neu

In [ ]:
# 4) ทำนาย sentiment
df_sentences["sentiment_pred"] = sentiment_batch_predict(
    text_list,

    
    batch_size=64,
    max_length=256
)

In [ ]:
# 5) แสดงตัวอย่างผลลัพธ์
print("===== ตัวอย่างผลลัพธ์การวิเคราะห์อารมณ์ (ระดับประโยค) =====")
display(
    df_sentences[
        ["sentence_clean", "sentiment_pred"]
    ].head(20)
)

จำนวนและ % ของแต่ละ sentiment

In [ ]:
# 6) สรุปจำนวนและสัดส่วน
print("===== จำนวนความคิดเห็นในแต่ละอารมณ์ =====")
sentiment_count = df_sentences["sentiment_pred"].value_counts()
display(sentiment_count)

print("===== สัดส่วนความคิดเห็น (%) =====")
sentiment_percent = (
    df_sentences["sentiment_pred"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
display(sentiment_percent)


In [ ]:
# 7) แปลง label สำหรับแสดงผลในรายงาน
label_map = {
    "pos": "Positive",
    "neg": "Negative",
    "neu": "Neutral"
}

df_sentences["sentiment_display"] = (
    df_sentences["sentiment_pred"]
    .map(label_map)
)

In [ ]:
# 8) บันทึกผลลัพธ์
df_sentences[
    ["sentence_clean", "sentiment_display"]
].rename(columns={"sentence_clean": "sentence"}).to_csv(
    "sentiment_results_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ บันทึกไฟล์ sentiment_results_final.csv เรียบร้อย")

# แสดงตัวอย่างสุดท้าย

display(
    df_sentences[
        ["sentence_clean", "sentiment_pred", "sentiment_display"]
    ].rename(columns={"sentence_clean": "sentence"})
    .head(20)
)


ดูว่าประเด็นถูกพูดถึงด้วยอารมณ์บวก ลบ หรือกลาง เท่าไหร่

In [ ]:
import pandas as pd

aspect_cols = ["System", "Logistics", "Company", "Store", "Promotion"]
sentiment_cols = ["Positive", "Negative", "Neutral"]

# เตรียม sentiment ให้เป็นชื่อมาตรฐาน
df_sentences["sentiment_display"] = df_sentences["sentiment_display"].fillna("Neutral")

# สร้าง matrix เปล่า
aspect_sentiment_matrix = pd.DataFrame(
    0, index=aspect_cols, columns=sentiment_cols
)

# นับค่า
for _, row in df_sentences.iterrows():
    for asp in aspect_cols:
        if row[f"RF_{asp}"] == 1:
            aspect_sentiment_matrix.loc[asp, row["sentiment_display"]] += 1

aspect_sentiment_matrix["Total"] = aspect_sentiment_matrix.sum(axis=1)
print("===== Aspect + Sentiment Matrix =====")
display(aspect_sentiment_matrix)


In [ ]:
pip install openpyxl

กำหนดรายชื่อประเด็น (aspects) ที่ต้องการใช้งาน

* เลือกเฉพาะคอลัมน์ที่จำเป็นจาก df_sentences ได้แก่ ปี เดือน ประโยคจริง อารมณ์ และผลการจำแนกประเด็นจาก Random Forest
* สร้าง DataFrame ใหม่สำหรับส่งออกข้อมูล

In [ ]:
aspect_cols = ["System", "Logistics", "Company", "Store", "Promotion"]

df_export = df_sentences[
    ["year", "month", "sentence_clean", "sentiment_display"]
    + [f"RF_{a}" for a in aspect_cols]
].copy()


# เปลี่ยนชื่อคอลัมน์ให้สวย
df_export = df_export.rename(columns={
    "sentence_clean": "Sentence",
    "sentiment_display": "Sentiment",
    **{f"RF_{a}": a for a in aspect_cols}
})


df_export.to_csv(
    "aspect_sentiment_per_sentence.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Export พร้อม year / month สำหรับ Power BI")


In [ ]:
import pandas as pd
from IPython.display import display

# อ่านไฟล์ที่ export สำหรับ Power BI
df_powerbi = pd.read_csv(
    "aspect_sentiment_per_sentence.csv",
    encoding="utf-8-sig"
)

print("✓ โหลดไฟล์ aspect_sentiment_per_sentence.csv สำเร็จ")
print("จำนวนแถว:", df_powerbi.shape[0])
print("จำนวนคอลัมน์:", df_powerbi.shape[1])

# แสดงตัวอย่างข้อมูล
display(df_powerbi.head(10))
